# Entrenamiento del Modelo MISR — Paso a Paso

Este notebook explica **en detalle** el proceso de entrenamiento del modelo **MISR (Multi-objective Iterated Symbolic Regression)** para la predicción de la Energía de Enlace Nuclear (BE). El flujo reproducido es el implementado en `run_comparison.py`, usando `MISR_Model` de `misr_advanced.py`.

---

### ¿Qué es MISR?

MISR es un algoritmo de **boosting simbólico iterativo** que construye una fórmula analítica final como suma de sub-expresiones matemáticas:

$$\hat{BE}(X) = f^{(1)}(X) + f^{(2)}(X) + \cdots + f^{(K)}(X)$$

Cada $f^{(k)}$ es un árbol de expresión simbólica encontrado por `gplearn` que ajusta el **residuo** de la iteración anterior, minimizando una función de pérdida **multiobjetivo** que combina:

- $L_{\text{main}}$: Error cuadrático ponderado sobre la Energía de Enlace (BE).
- $L_{\text{aux}}$: Consistencia con variables físicas derivadas: $S_n$, $S_{2n}$, $S_p$, $S_{2p}$, $BE/A$.
- $L_{\text{penalty}}$: Penalización de derivadas numéricas para garantizar suavidad de la fórmula.

## Paso 1 — Importación de Librerías

El modelo **no usa TensorFlow ni PyTorch**. El stack tecnológico es:

| Librería | Rol |
|----------|-----|
| `misr_advanced.MISR_Model` | Motor principal del algoritmo MISR |
| `gplearn` | Motor de regresión simbólica (evolución de expresiones matemáticas) |
| `scikit-learn` | Evaluación de importancia de features (GBT), K-Fold CV y métricas |
| `nuclearpy_models.sr_be` | Modelo de referencia (baseline) basado en regresión simbólica clásica |
| `numpy / pandas` | Manipulación numérica y de datos |

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Modelo MISR avanzado
from misr_advanced import MISR_Model

# Modelo base de referencia
sys.path.append(os.path.abspath('nuclearpy_models'))
from models.BE.sr import sr_be

np.random.seed(42)
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## Paso 2 — Carga de Datos

Los datos provienen de medidas experimentales de masas nucleares (tabla AME). Están divididos en:

- **`be_train.csv`**: Núcleos usados para entrenamiento.
- **`be_test.csv`**: Núcleos reservados para evaluación final.

El dataset contiene la **Energía de enlace** (`BE`), la incertidumbre experimental (`uBE`) y las variables que definen propiamente al núcleo (`N` y `Z`).

In [2]:
df_train = pd.read_csv("Data/Experimental/be_train.csv")
df_test  = pd.read_csv("Data/Experimental/be_test.csv")

print(f"Muestras de entrenamiento : {len(df_train)}")
print(f"Muestras de prueba        : {len(df_test)}")
df_train.head()

Muestras de entrenamiento : 2749
Muestras de prueba        : 688


,BE,uBE,N,Z,eBE,x,A,ee,eo,oe,...,h,P,K,S,d,sp,sn,s2p,s2n,be_per_A
0,983.318392,0.025553,64,53,0,121,117,0.0,0.0,1.0,...,0.828125,2.470588,4.890973,0.094017,0.0,4.929472,8.603803,6.444437,19.481804,8.404431
1,1219.577365,0.011189,90,58,0,1024,148,1.0,0.0,0.0,...,0.644444,4.000000,5.289572,0.216216,1.0,7.929931,4.342816,17.859470,10.591085,8.240388
2,631.329222,0.001861,43,30,0,169,73,0.0,1.0,0.0,...,0.697674,1.555556,4.179339,0.178082,0.0,9.745181,8.234626,20.841498,13.108136,8.648345
3,1143.905000,0.145000,76,69,1,49,145,0.0,0.0,1.0,...,0.907895,4.105263,5.253588,0.048276,0.0,0.135980,11.539000,-2.043990,24.571217,7.889000
4,699.300000,0.336000,42,42,1,0,84,1.0,0.0,0.0,...,1.000000,4.000000,4.379519,0.000000,1.0,-1.025000,11.413220,0.138000,26.085705,8.325000


## Paso 3 — Ingeniería de Características (Las 7 variables físicas del MISR)

A partir de los parámetros nativos de cada núcleo (`N` neutrones, y `Z` protones), derivamos **5 variables físicas adicionales** que capturan fenómenos macroscópicos y de capa cuántica. Al finalizar, obtendremos exactamente las **7 variables** de la arquitectura MISR.

| Variable | Descripción Físico-Teórica | Cálculo |
|----------|-------------|---------|
| `N` | Número de neutrones. | - |
| `Z` | Número de protones. | - |
| `A` | Número másico. | $A = N + Z$ |
| `I` | Isospín. | $I = (N-Z)/A$ |
| `P` | Factor de valencia n-p. | $\frac{N_n \cdot N_p}{N_n + N_p}$ |
| `Nn` | Cierre de capa (neutrones). | Distancia al número mágico más cercano. |
| `Np` | Cierre de capa (protones). | Distancia al número mágico más cercano. |

In [3]:
def prepare_features(df):
    df = df.copy()
    
    # Masa Total e Isospín
    df['A'] = df['N'] + df['Z']
    df['I'] = (df['N'] - df['Z']) / df['A']

    # Distancias a Cierres de Capas (Números Mágicos)
    z_magic = np.array([2, 8, 20, 28, 50, 82, 126])
    n_magic = np.array([2, 8, 20, 28, 50, 82, 126, 184])
    
    df['Np'] = np.min(np.abs(df['Z'].values[:, None] - z_magic[None, :]), axis=1)
    df['Nn'] = np.min(np.abs(df['N'].values[:, None] - n_magic[None, :]), axis=1)

    # Factor P (Interacción protón-neutrón empírica)
    df['P'] = np.where((df['Nn'] + df['Np']) > 0, 
                       (df['Nn'] * df['Np']) / (df['Nn'] + df['Np']), 0.0)

    # El modelo extrae internamente la incertidumbre experimental bajo este nombre clave:
    if 'uBE' in df.columns:
        df.rename(columns={'uBE': 'bindingEnergyUncertainty'}, inplace=True)
        
    return df

df_train = prepare_features(df_train)
df_test  = prepare_features(df_test)

print("✓ Las 7 features maestras han sido calculadas.")

# Comprobamos exclusivamente las variables estrictas que alimentarán el motor:
df_train[['N', 'Z', 'A', 'I', 'P', 'Nn', 'Np']].head(5)

✓ Las 7 features maestras han sido calculadas.


,N,Z,A,I,P,Nn,Np
0,64,53,117,0.094017,2.470588,14,3
1,90,58,148,0.216216,4.000000,8,8
2,43,30,73,0.178082,1.555556,7,2
3,76,69,145,0.048276,4.105263,6,13
4,42,42,84,0.000000,4.000000,8,8


## Paso 4 — Inicialización de `MISR_Model`

Al preparar la Regresión Simbólica establecemos restricciones de costo vs poder representativo:
- `maxiter`: Iteraciones de boosting asintóticas (se suman iteraciones ajustando sobre el residuo subyacente de BE).
- `k_folds`: División interna para validación en la estructura Mega-Matriz (Fold K para evitar sobreajuste local).
- `s_features`: Variables base sobre de las $7$ introducidas que cada fold puede utilizar por evolución mutacional.
- `n_generations`: Extensión temporal evolutiva `gplearn`.

In [6]:
misr = MISR_Model(
    maxiter=10,
    theta = -1,
    k_folds=5,
    s_features=4,
    n_generations=50
)

print("Instancia de MISR generada correctamente.")

Instancia de MISR generada correctamente.


## Paso 5 — Ejecución del Entrenamiento (`fit`)

El algoritmo compila y entrena en el siguiente flujo:
1. Extrae únicamente las 7 features calculadas `['N', 'Z', 'A', 'I', 'P', 'Nn', 'Np']`.
2. Amplia esas representaciones creando combinaciones $N-1$, $N-2$, $Z+1$, $Z+2$ (Vecinos Mega-X).
3. Aplica regresión iterativa de Gradient Boosting internamente.
4. Almacena cada fórmula que reduzca la Función de Pérdida en validación multiobjetivo por abajo del umbral $\theta$.

In [7]:
print("Entrenando el modelo MISR...")
start = time.time()

misr.fit(df_train, target_col='BE')

print(f"\nEntrenamiento Finalizado en {time.time() - start:.2f} s")
print(f"El modelo almacenó {len(misr.models)} sub-expresiones de expansión iterativa.")

Entrenando el modelo MISR...
Iniciando Entrenamiento MISR Multiobjetivo...

--- Iteración 1/10 ---
  Mejor Ecuación: add(add(div(N, 0.207), add(A, div(div(mul(add(Z, add(Z, Z)), add(A, Z)), add(Z, div(N, N))), 0.926))), sub(sub(sub(sub(add(Z, Z), div(sub(sub(sub(N, Z), sub(Np, N)), sub(sub(sub(sub(sub(add(Z, Z), div(N, A)), div(sub(div(A, N), sub(Np, N)), sub(A, sub(sub(N, Z), sub(N, 0.621))))), sub(sub(N, Z), sub(Np, N))), Np), sub(sub(N, div(sub(div(A, N), sub(Np, N)), sub(A, sub(add(Z, div(A, N)), sub(N, Z))))), sub(Np, N)))), div(div(mul(add(Z, div(div(add(Z, Z), div(N, A)), div(sub(Z, sub(Z, N)), add(Z, div(A, N))))), add(Z, div(A, N))), div(div(mul(sub(sub(N, Z), sub(Np, N)), div(N, 0.207)), add(Z, div(A, N))), 0.926)), 0.926))), sub(div(sub(sub(sub(sub(sub(add(sub(sub(sub(add(Z, Z), div(A, N)), sub(sub(N, Z), sub(sub(sub(sub(sub(sub(add(Z, Z), div(div(N, 0.207), sub(A, sub(sub(N, Z), sub(Np, N))))), div(add(Z, Z), sub(A, sub(sub(N, Z), sub(Np, N))))), sub(sub(N, Z), sub(Np, N)))

## Paso 6 — Interpretando la Fórmula Analítica

A diferencia del Deep Learning, aquí observamos directamente la arquitectura logica de la fórmula final aprendida `misr.get_formula()`, la cual es simple combinatoria algebraica transparente sumada a los sub-términos de Boosting.

In [8]:
print("\n--- Expresión Analítica del MISR ---")
print(misr.get_formula())

print("\n--- Términos Aprendidos ---")
for i, model_info in enumerate(misr.models):
    print(f"Iteración {i+1}:")
    print(f"  Fórmula: {model_info['model']._program}")
    print(f"  Pérdida (Loss) Valida: {model_info['loss']:.4f}")
    print(f"  Features Activas:      {model_info['features_names']}")


--- Expresión Analítica del MISR ---
(add(add(div(N, 0.207), add(A, div(div(mul(add(Z, add(Z, Z)), add(A, Z)), add(Z, div(N, N))), 0.926))), sub(sub(sub(sub(add(Z, Z), div(sub(sub(sub(N, Z), sub(Np, N)), sub(sub(sub(sub(sub(add(Z, Z), div(N, A)), div(sub(div(A, N), sub(Np, N)), sub(A, sub(sub(N, Z), sub(N, 0.621))))), sub(sub(N, Z), sub(Np, N))), Np), sub(sub(N, div(sub(div(A, N), sub(Np, N)), sub(A, sub(add(Z, div(A, N)), sub(N, Z))))), sub(Np, N)))), div(div(mul(add(Z, div(div(add(Z, Z), div(N, A)), div(sub(Z, sub(Z, N)), add(Z, div(A, N))))), add(Z, div(A, N))), div(div(mul(sub(sub(N, Z), sub(Np, N)), div(N, 0.207)), add(Z, div(A, N))), 0.926)), 0.926))), sub(div(sub(sub(sub(sub(sub(add(sub(sub(sub(add(Z, Z), div(A, N)), sub(sub(N, Z), sub(sub(sub(sub(sub(sub(add(Z, Z), div(div(N, 0.207), sub(A, sub(sub(N, Z), sub(Np, N))))), div(add(Z, Z), sub(A, sub(sub(N, Z), sub(Np, N))))), sub(sub(N, Z), sub(Np, N))), sub(N, 0.621)), Np), sub(sub(N, sub(-0.670, Np)), sub(Np, N))))), Np), add(A

## Paso 7 — Inferencia y Evaluación

Ahora filtramos únicamente el set de test a núcleos con $12 \leq Z \leq 50$. Posteriormente medimos su RMSE, MAE y $R^2$ contra un modelo referencial estándar `nuclearpy` (un ajuste Liquid Drop generalizado de ecuaciones empíricas).

In [10]:
df_test_filt = df_test[(df_test['Z'] >= 12) & (df_test['Z'] <= 50)].copy()
df_test_filt = df_test_filt.dropna(subset=['BE']).reset_index(drop=True)

y_true = df_test_filt['BE'].values

y_pred_misr = misr.predict(df_test_filt)

y_pred_base = []
for z, n in zip(df_test_filt['Z'], df_test_filt['N']):
    pred, _ = sr_be(z, n)
    y_pred_base.append(pred)
y_pred_base = np.array(y_pred_base)

rmse_m = np.sqrt(mean_squared_error(y_true, y_pred_misr))
mae_m  = mean_absolute_error(y_true, y_pred_misr)
r2_m   = r2_score(y_true, y_pred_misr)

rmse_b = np.sqrt(mean_squared_error(y_true, y_pred_base))
mae_b  = mean_absolute_error(y_true, y_pred_base)
r2_b   = r2_score(y_true, y_pred_base)

print(f"--- RESULTADOS (Test {len(y_true)} núcleos Z ∈ [12, 50]) ---")
print(f"{'Modelo':<20} | {'RMSE [MeV]':<15} | {'MAE [MeV]':<15} | {'R2':<10}")
print("-"*65)
print(f"{'MISR (Ours)':<20} | {rmse_m:<15.4f} | {mae_m:<15.4f} | {r2_m:<10.4f}")
print(f"{'SR Base (nuclearpy)':<20} | {rmse_b:<15.4f} | {mae_b:<15.4f} | {r2_b:<10.4f}")

--- RESULTADOS (Test 274 núcleos Z ∈ [12, 50]) ---
Modelo               | RMSE [MeV]      | MAE [MeV]       | R2        
-----------------------------------------------------------------
MISR (Ours)          | 6.2123          | 3.9786          | 0.9993    
SR Base (nuclearpy)  | 1.0580          | 0.8290          | 1.0000    


## Paso 8 — Exportación de Resultados

Por finalización documentamos y guardamos la regresión del test de evaluación completa.

In [11]:
df_results = pd.DataFrame({
    'N'               : df_test_filt['N'],
    'Z'               : df_test_filt['Z'],
    'A'               : df_test_filt['A'],
    'True_BE_MeV'     : y_true,
    'MISR_Pred_MeV'   : y_pred_misr,
    'MISR_Error'      : y_true - y_pred_misr,
    'Base_SR_Pred_MeV': y_pred_base,
    'Base_SR_Error'   : y_true - y_pred_base
})

csv_path = "comparison_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"Tabla exportada a test-run: {csv_path}")
df_results.head()

Tabla exportada a test-run: comparison_results.csv


,N,Z,A,True_BE_MeV,MISR_Pred_MeV,MISR_Error,Base_SR_Pred_MeV,Base_SR_Error
0,39,41,80,652.080000,659.987825,-7.907825,651.451545,0.628455
1,25,18,43,364.994243,371.352256,-6.358014,366.478823,-1.484581
2,49,49,98,806.540000,813.117010,-6.577010,807.190635,-0.650635
3,28,30,58,486.964909,483.839186,3.125723,486.579267,0.385642
4,38,30,68,595.386376,592.118239,3.268137,596.546741,-1.160365
